# 03 — Menilai dataset

Notebook ini menjawab satu pertanyaan: **dataset ini layak dipakai melatih model
auto-description dan rekomendasi harga, atau belum?**

Bukan "berapa banyak baris". Baris banyak tapi deskripsinya kosong semua tidak
ada gunanya untuk auto-description.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tokopedia_scraper.config import Config
from tokopedia_scraper.storage import Storage

cfg = Config.load(ROOT / "config.yaml")

with Storage(cfg.storage.db_path) as store:
    df = pd.read_sql_query("SELECT * FROM products", store.conn)

for column in ("category_path", "image_urls", "local_image_paths"):
    df[column] = df[column].apply(lambda v: json.loads(v) if isinstance(v, str) and v else [])
df["specs"] = df["specs"].apply(lambda v: json.loads(v) if isinstance(v, str) and v else {})

df["description"] = df["description"].fillna("")
df["desc_len"] = df["description"].str.len()
df["n_images"] = df["image_urls"].apply(len)
df["n_local_images"] = df["local_image_paths"].apply(len)
df["n_specs"] = df["specs"].apply(len)
df["category"] = df["category_path"].apply(lambda p: p[-1] if p else "(unknown)")

print(f"{len(df):,} produk, {int(df['pdp_fetched'].sum()):,} sudah di-enrich")
df.head(3)[["product_id", "title", "price", "rating", "desc_len", "n_images"]]

## 1. Kelengkapan per kolom

Kolom yang kosong di **semua** baris berarti parser meleset, bukan Tokopedia yang
pelit. Kolom yang kosong sebagian biasanya wajar.

In [ ]:
completeness = pd.DataFrame({"terisi": df.notna().sum(), "kosong": df.isna().sum()})
completeness["terisi_%"] = (100 * completeness["terisi"] / max(len(df), 1)).round(1)
display(completeness.sort_values("terisi_%"))

empty_everywhere = completeness[completeness["terisi"] == 0].index.tolist()
if empty_everywhere:
    print(f"\nKOSONG DI SEMUA BARIS -> curigai parser: {empty_everywhere}")

## 2. Deskripsi — inti untuk auto-description

Tiga kelompok yang wajib dibedakan:

| Kelompok | Artinya |
|---|---|
| belum di-enrich | stage 2 belum dijalankan untuk produk ini |
| deskripsi berupa gambar | penjual mengunggah deskripsi sebagai gambar; tidak ada teks sama sekali |
| terlalu pendek | ada teks tapi di bawah ambang minimum untuk jadi target latih |

In [ ]:
MIN_USABLE_CHARS = 120

enriched = df[df["pdp_fetched"] == 1]
not_enriched = len(df) - len(enriched)
image_only = int(((enriched["desc_len"] == 0) & (enriched["n_specs"] > 0)).sum())
no_text_at_all = int(((enriched["desc_len"] == 0) & (enriched["n_specs"] == 0)).sum())
too_short = int(
    ((enriched["desc_len"] > 0) & (enriched["desc_len"] < MIN_USABLE_CHARS)).sum()
)
usable = int((enriched["desc_len"] >= MIN_USABLE_CHARS).sum())

display(pd.Series({
    "total produk": len(df),
    "belum di-enrich": not_enriched,
    "deskripsi berupa gambar": image_only,
    "tanpa teks & tanpa specs": no_text_at_all,
    f"teks < {MIN_USABLE_CHARS} char": too_short,
    f"LAYAK LATIH (>= {MIN_USABLE_CHARS})": usable,
}).to_frame("jumlah"))

if len(df):
    print(f"\nHasil bersih: {usable:,} dari {len(df):,} produk "
          f"({usable / len(df):.1%}) punya deskripsi teks yang cukup panjang.")

In [ ]:
with_text = enriched[enriched["desc_len"] > 0]["desc_len"]

if len(with_text):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(with_text, bins=50, color="#3b7dd8", edgecolor="white")
    axes[0].axvline(MIN_USABLE_CHARS, color="crimson", linestyle="--",
                    label=f"ambang {MIN_USABLE_CHARS}")
    axes[0].set_title("Panjang deskripsi (yang punya teks)")
    axes[0].set_xlabel("karakter")
    axes[0].set_ylabel("produk")
    axes[0].legend()

    axes[1].hist(with_text, bins=50, color="#3b7dd8", edgecolor="white")
    axes[1].set_xscale("log")
    axes[1].set_title("Sama, skala log — melihat ekor panjang")
    axes[1].set_xlabel("karakter (log)")

    plt.tight_layout()
    plt.show()
    print(with_text.describe().round(0).to_string())
else:
    print("Belum ada deskripsi. Jalankan `python main.py enrich` dulu.")

## 3. Harga per kategori — inti untuk rekomendasi harga

Skala log, karena harga marketplace membentang beberapa orde besaran dan skala
linear hanya akan menampilkan satu batang.

In [ ]:
priced = df[df["price"].notna() & (df["price"] > 0)]
print(f"{len(priced):,} produk punya harga")

if len(priced):
    top = priced["category"].value_counts().head(8).index.tolist()
    subset = priced[priced["category"].isin(top)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(priced["price"], bins=60, color="#2e9e6b", edgecolor="white")
    axes[0].set_xscale("log")
    axes[0].set_title("Distribusi harga (semua produk)")
    axes[0].set_xlabel("harga (Rp, log)")
    axes[0].set_ylabel("produk")

    if len(subset):
        data = [subset[subset["category"] == c]["price"].values for c in top]
        axes[1].boxplot(data, tick_labels=[c[:16] for c in top])
        axes[1].set_yscale("log")
        axes[1].set_title("Harga per kategori")
        axes[1].set_ylabel("harga (Rp, log)")
        axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

    per_cat = priced.groupby("category")["price"].agg(["count", "median", "min", "max"])
    display(per_cat.sort_values("count", ascending=False).head(15))

### Harga tidak masuk akal

Ditemukan pada data sungguhan: sebuah **kaos** dihargai Rp1.999.960.000, dan ada
sandal berjudul "sendal jepit mahal, murah meriah jgn dibeli". Response mentahnya
memang berisi angka itu — jadi ini bukan parser yang meleset, melainkan penjual
yang iseng atau salah ketik nol.

Untuk model rekomendasi harga, baris seperti ini merusak. Deteksinya per kategori,
bukan global: Rp5 juta wajar untuk laptop, tidak wajar untuk kaos kaki. Metodenya
MAD (median absolute deviation) pada log-harga — tahan terhadap outlier, tidak
seperti mean dan standar deviasi yang justru ditarik oleh outlier itu sendiri.

In [ ]:
import numpy as np

MAD_THRESHOLD = 4.0    # modified z-score; 3.5 lazim dipakai, 4.0 lebih longgar
MIN_PER_CATEGORY = 20  # di bawah ini statistiknya tidak bermakna

df["price_outlier"] = False

if len(priced):
    log_price = np.log10(priced["price"])

    for category, index in priced.groupby("category").groups.items():
        if len(index) < MIN_PER_CATEGORY:
            continue
        values = log_price.loc[index]
        median = values.median()
        mad = (values - median).abs().median()
        if mad == 0:
            continue
        # 0.6745 menormalkan MAD agar setara standar deviasi pada distribusi normal
        modified_z = 0.6745 * (values - median).abs() / mad
        df.loc[index[modified_z > MAD_THRESHOLD], "price_outlier"] = True

# priced diambil sebelum kolom di atas ada, jadi harus disegarkan.
priced = df[df["price"].notna() & (df["price"] > 0)]

n_outlier = int(df["price_outlier"].sum())
print(f"harga tidak masuk akal: {n_outlier:,} dari {len(priced):,} "
      f"({n_outlier / max(len(priced), 1):.2%})")

if n_outlier:
    display(
        df[df["price_outlier"]]
        .sort_values("price", ascending=False)[
            ["price", "category", "source_keyword", "title"]
        ]
        .head(12)
    )

clean = priced[~priced["price_outlier"]]
if len(clean):
    print(f"\nsetelah dibuang: {len(clean):,} produk")
    print(f"  median  Rp{clean['price'].median():,.0f}")
    print(f"  min     Rp{clean['price'].min():,.0f}")
    print(f"  max     Rp{clean['price'].max():,.0f}")
    print(f"\nsebelum dibuang max-nya Rp{priced['price'].max():,.0f}")

print("\nCatatan: kolom `price_outlier` hanya ada di DataFrame notebook ini,")
print("tidak ditulis ke database. Dataset mentah sengaja dibiarkan apa adanya —")
print("penyaringan adalah keputusan tahap training, bukan tahap scraping.")

## 4. Gambar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

max_urls = int(df["n_images"].max() or 1)
max_local = int(df["n_local_images"].max() or 1)

axes[0].hist(df["n_images"], bins=range(0, max_urls + 2),
             color="#c46a2f", edgecolor="white", align="left")
axes[0].set_title("URL gambar per produk")
axes[0].set_xlabel("jumlah URL")
axes[0].set_ylabel("produk")

axes[1].hist(df["n_local_images"], bins=range(0, max_local + 2),
             color="#7a51b5", edgecolor="white", align="left")
axes[1].set_title("Gambar yang benar-benar terunduh")
axes[1].set_xlabel("jumlah berkas")

plt.tight_layout()
plt.show()

print(f"tanpa URL gambar   : {int((df['n_images'] == 0).sum()):,}")
print(f"belum diunduh      : {int((df['n_local_images'] == 0).sum()):,}")
print(f"rata-rata URL      : {df['n_images'].mean():.1f}")
print(f"rata-rata terunduh : {df['n_local_images'].mean():.1f}")

## 5. Duplikat

`product_id` adalah primary key, jadi duplikat persis mustahil. Yang perlu
diwaspadai adalah **produk sama yang di-listing ulang dengan id berbeda** —
sangat umum di Tokopedia, dan membuat model melihat contoh yang sama berkali-kali.

In [ ]:
assert df["product_id"].duplicated().sum() == 0, "primary key gagal — cek storage.py"

titled = df[df["title"].notna()]
dup_title = int(titled.duplicated(subset=["title"], keep=False).sum())
dup_title_shop = int(titled.duplicated(subset=["title", "shop_id"], keep=False).sum())
dup_url = int(df.duplicated(subset=["url"], keep=False).sum())
dup_desc = int(df[df["desc_len"] > 0].duplicated(subset=["description"], keep=False).sum())

print(f"judul identik         : {dup_title:,}")
print(f"judul + toko identik  : {dup_title_shop:,}   <- hampir pasti listing ulang")
print(f"URL identik           : {dup_url:,}")
print(f"deskripsi identik     : {dup_desc:,}   <- penting: bocor antara train dan test")

if dup_title:
    display(
        titled[titled.duplicated(subset=["title"], keep=False)]
        .sort_values("title")[["product_id", "shop_name", "title", "price"]]
        .head(10)
    )

## 6. Sebaran keyword dan toko

Kalau satu toko mendominasi, model akan belajar gaya penulisan toko itu, bukan
gaya deskripsi produk pada umumnya.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

by_keyword = df["source_keyword"].value_counts()
by_keyword.head(20).plot.barh(ax=axes[0], color="#3b7dd8")
axes[0].set_title("Produk per keyword (20 teratas)")
axes[0].invert_yaxis()

by_shop = df["shop_name"].value_counts()
by_shop.head(15).plot.barh(ax=axes[1], color="#c46a2f")
axes[1].set_title("Produk per toko (15 teratas)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

shop_share = (by_shop.iloc[0] / max(len(df), 1)) if len(by_shop) else 0.0
print(f"{df['source_keyword'].nunique()} keyword, {df['shop_id'].nunique():,} toko berbeda")
print(f"toko terbesar menyumbang {shop_share:.2%} dari seluruh dataset")
if shop_share > 0.15:
    print("^ terlalu terkonsentrasi. Tambah keyword agar lebih beragam.")

## 7. Vonis

In [ ]:
from datetime import datetime, timezone

n = max(len(df), 1)
# bool() on each: pandas comparisons return numpy.bool_, and summing those
# gives a numpy.int64 that json.dumps refuses to serialise.
checks = {
    "minimal 5.000 produk": bool(len(df) >= 5_000),
    f"minimal 3.000 deskripsi >= {MIN_USABLE_CHARS} char": bool(usable >= 3_000),
    "minimal 80% punya harga": bool(len(priced) >= 0.8 * n),
    "minimal 5 kategori berbeda": bool(df["category"].nunique() >= 5),
    "tidak ada toko > 15% dataset": bool(shop_share <= 0.15),
    "duplikat deskripsi < 10%": bool(dup_desc < 0.1 * n),
    # Ambang 5%, bukan 0%: sebagian yang tertandai memang barang mahal yang sah.
    # Yang dijaga di sini adalah "harga tidak didominasi sampah", bukan "nol outlier".
    "harga outlier < 5%": bool(n_outlier < 0.05 * max(len(priced), 1)),
}

# Obat per kriteria. Pesan generik "enrich belum selesai" pernah muncul saat
# enrich justru sudah 100% — menuduh penyebab yang salah lebih buruk daripada
# tidak menuduh sama sekali.
REMEDY = {
    "minimal 5.000 produk": "tambah keyword di config.yaml, lalu `python main.py search`",
    f"minimal 3.000 deskripsi >= {MIN_USABLE_CHARS} char":
        "jalankan `python main.py enrich` sampai pending_pdp = 0",
    "minimal 80% punya harga": "cek parsers.py — harga seharusnya selalu ada dari stage 1",
    "minimal 5 kategori berbeda": "keyword terlalu sempit; sebar ke kategori lain",
    "tidak ada toko > 15% dataset": "tambah keyword agar tidak didominasi satu toko",
    "duplikat deskripsi < 10%":
        "wajar di marketplace (listing ulang + reseller). Jangan dibuang — "
        "pisahkan train/test per-grup deskripsi (GroupShuffleSplit) supaya "
        "contoh yang sama tidak bocor ke kedua sisi",
    "harga outlier < 5%": "saring per kategori sebelum melatih model harga",
}

for label, ok in checks.items():
    print(f"  [{'OK' if ok else '--'}] {label}")

passed = int(sum(checks.values()))
print(f"\n{passed}/{len(checks)} kriteria terpenuhi")

failed = [label for label, ok in checks.items() if not ok]
if failed:
    print("\nYang belum terpenuhi, dan apa yang harus dilakukan:")
    for label in failed:
        print(f"  - {label}")
        print(f"      -> {REMEDY.get(label, 'periksa manual')}")
else:
    print("Semua kriteria terpenuhi.")

print("\nCatatan soal outlier: detektor MAD ini lolos-lolosan untuk kategori yang")
print("sebarannya sangat lebar. Sebelum melatih model harga, saring lagi per")
print("kategori dengan batas yang kamu tentukan sendiri.")

summary = {
    "products": int(len(df)),
    "enriched": int(len(enriched)),
    "usable_for_training": int(usable),
    "image_only_description": int(image_only),
    "median_description_chars": float(with_text.median()) if len(with_text) else 0.0,
    "unique_descriptions": int(df.loc[df["desc_len"] > 0, "description"].nunique()),
    "categories": int(df["category"].nunique()),
    "shops": int(df["shop_id"].nunique()),
    "priced": int(len(priced)),
    "price_outliers": int(n_outlier),
    "duplicate_titles": int(dup_title),
    "duplicate_descriptions": int(dup_desc),
    "largest_shop_share": round(float(shop_share), 4),
    "checks_passed": passed,
    "checks_total": len(checks),
    "checks_failed": failed,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}

out = cfg.storage.export_dir / "eda_summary.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\nringkasan ditulis: {out}")